# OMTF HECIN Testset Analysis

Validates the 1000-event testset files for all 8 datasets.

| Tag | Description | pT range | Displacement |
|-----|-------------|----------|--------------|
| S1  | Single prompt muon, no PU | 2–200 GeV | — |
| S2  | Single displaced muon, no PU | 2–200 GeV | flat |dxy| ≤ 50 cm |
| S3  | DiMuon same sector, no PU | 2–100 GeV | — |
| S4  | TriMuon same sector, no PU | 5–80 GeV | — |
| S5  | Two displaced muons same sector, no PU | 5–100 GeV | flat |dxy| ≤ 30 cm |
| B1  | Single muon + PU200 | 2–200 GeV | — |
| B2  | Displaced muon + PU200 | 2–200 GeV | flat |dxy| ≤ 50 cm |
| B3  | DiMuon + PU200 | 2–100 GeV | — |

In [2]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

ROOT.gROOT.SetBatch(True)
ROOT.gErrorIgnoreLevel = ROOT.kError

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

EOS_BASE  = '/eos/user/p/pleguina/omtf_hecin_datasets/testset'
TREE_PATH = 'simOmtfPhase2Digis/OMTFHitsTree'

# (tag, label, color, linestyle, category)
DATASETS = [
    ('S1', 'S1 single \u03bc',               'tab:blue',   '-',  'signal'),
    ('S2', 'S2 displ. \u03bc (|dxy|\u226450cm)',  'tab:orange', '-',  'signal_disp'),
    ('S3', 'S3 di-\u03bc',                   'tab:green',  '-',  'signal'),
    ('S4', 'S4 tri-\u03bc',                  'tab:red',    '-',  'signal'),
    ('S5', 'S5 2-displ. \u03bc (|dxy|\u226430)',  'tab:purple', '-',  'signal_disp'),
    ('B1', 'B1 \u03bc + PU200',              'tab:blue',   '--', 'bkg'),
    ('B2', 'B2 displ. \u03bc + PU200',       'tab:orange', '--', 'bkg_disp'),
    ('B3', 'B3 di-\u03bc + PU200',           'tab:green',  '--', 'bkg'),
]
DS_META = {t: (lbl, col, ls, cat) for t, lbl, col, ls, cat in DATASETS}


ModuleNotFoundError: No module named 'JupyROOT'

In [ ]:
# ── Load all datasets ──────────────────────────────────────────────────────
# Branch types (from TLeaf::GetTypeName()):
#   Float_t / Short_t / UInt_t / Bool_t → getattr works fine
#   Char_t → getattr fails for negative values; use GetLeaf().GetValue()
#
CHAR_BRANCHES  = ['muonCharge', 'omtfCharge', 'omtfProcessor',
                  'omtfQuality', 'omtfRefLayer', 'omtfRefHitNum']
FLOAT_BRANCHES = ['muonPt', 'muonEta', 'muonPhi', 'muonPropEta', 'muonPropPhi',
                  'muonDxy', 'muonRho', 'vertexEta', 'vertexPhi',
                  'omtfPt', 'omtfUPt', 'omtfEta', 'omtfPhi',
                  'deltaEta', 'deltaPhi']
INT_BRANCHES   = ['eventNum', 'muonEvent', 'parentPdgId', 'omtfHwEta',
                  'omtfScore', 'omtfRefHitPhi', 'omtfFiredLayers']
BOOL_BRANCHES  = ['killed']


def load_dataset(tag):
    path = f'{EOS_BASE}/{tag}/omtf_hits_{tag}_0.root'
    tf = ROOT.TFile.Open(path)
    if not tf or tf.IsZombie():
        return None
    tree = tf.Get(TREE_PATH)
    if not tree:
        return None

    # Pre-fetch leaf objects for Char_t branches (getattr broken for signed negatives)
    char_leaves = {b: tree.GetLeaf(b) for b in CHAR_BRANCHES}

    arrays = defaultdict(list)
    for i in range(tree.GetEntries()):
        tree.GetEntry(i)
        for b in FLOAT_BRANCHES:
            arrays[b].append(float(getattr(tree, b)))
        for b in CHAR_BRANCHES:
            arrays[b].append(int(char_leaves[b].GetValue()))
        for b in INT_BRANCHES:
            arrays[b].append(int(getattr(tree, b)))
        for b in BOOL_BRANCHES:
            arrays[b].append(bool(getattr(tree, b)))
        arrays['nhits'].append(len(tree.hits_r))

    return {k: np.array(v) for k, v in arrays.items()}


data = {}
for tag, *_ in DATASETS:
    print(f'Loading {tag}...', end=' ', flush=True)
    d = load_dataset(tag)
    if d is None:
        print('MISSING')
        continue
    data[tag] = d
    n = len(d['muonPt'])
    print(f'{n} entries | muonPt mean={d["muonPt"].mean():.1f} GeV | '
          f'|dxy| mean={np.abs(d["muonDxy"]).mean():.3f} cm | '
          f'omtfPt>0 = {np.mean(d["omtfPt"]>0)*100:.0f}%')

LOADED = [t for t, *_ in DATASETS if t in data]
print(f'\nLoaded {len(LOADED)}/8 datasets: {LOADED}')


## 1. Dataset overview — entry counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

tags   = [t for t, *_ in DATASETS if t in data]
labels = [DS_META[t][0] for t in tags]
colors = [DS_META[t][1] for t in tags]
counts = [len(data[t]['muonPt']) for t in tags]

bars = ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=0.7)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_ylabel('OMTF candidates (tree entries)')
ax.set_title('Testset: entries per dataset (1000 events generated each)')
ax.set_ylim(0, max(counts) * 1.18)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('testset_01_counts.pdf', bbox_inches='tight')
plt.show()
print('Note: entries < 1000 because not every event yields an OMTF candidate in 0.82<|\u03b7|<1.24')

## 2. Generator-level muon pT

**Expected**: 1/pT-flat gun → the pT spectrum falls as 1/pT (constant dN/d(1/pT)).  
**Check**: plot in linear pT — should look like 1/pT power law. Also plot 1/pT which should be flat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins_pt    = np.linspace(0, 220, 45)
bins_invpt = np.linspace(0, 0.55, 45)

for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    pt = data[tag]['muonPt']
    axes[0].hist(pt, bins=bins_pt, histtype='step', label=lbl,
                 color=col, linestyle=ls, linewidth=1.6, density=True)
    axes[1].hist(1.0/pt, bins=bins_invpt, histtype='step', label=lbl,
                 color=col, linestyle=ls, linewidth=1.6, density=True)

axes[0].set_xlabel(r'Gen $p_T$ [GeV]')
axes[0].set_ylabel('Density')
axes[0].set_title(r'Gen $p_T$ (should fall as $1/p_T$)')
axes[0].legend(ncol=2)

axes[1].set_xlabel(r'Gen $1/p_T$ [GeV$^{-1}$]')
axes[1].set_ylabel('Density')
axes[1].set_title(r'Gen $1/p_T$ (should be flat)')

plt.tight_layout()
plt.savefig('testset_02_muonPt.pdf', bbox_inches='tight')
plt.show()

## 3. Generator-level muon η

**Expected**: all events in OMTF acceptance: 0.82 < |η| < 1.24

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, branch, title in [
    (axes[0], 'muonEta',     r'Gen $\eta$ (truth vertex)'),
    (axes[1], 'muonPropEta', r'Gen $\eta$ (propagated to OMTF)'),
]:
    for tag in LOADED:
        lbl, col, ls, _ = DS_META[tag]
        ax.hist(data[tag][branch], bins=50, range=(-1.6, 1.6),
                histtype='step', label=lbl,
                color=col, linestyle=ls, linewidth=1.6, density=True)
    for xv in [0.82, 1.24, -0.82, -1.24]:
        ax.axvline(xv, color='dimgray', linestyle=':', linewidth=1.0, alpha=0.8)
    ax.set_xlabel(r'$\eta$')
    ax.set_ylabel('Density')
    ax.set_title(title)

axes[0].legend(ncol=2)
plt.tight_layout()
plt.savefig('testset_03_eta.pdf', bbox_inches='tight')
plt.show()

## 4. Displacement variables: dxy and ρ

**Expected**:
- S1, S3, S4, B1, B3: prompt — `muonDxy ≈ 0`, `muonRho ≈ 0`
- S2, B2: flat `|dxy|` up to 50 cm, `muonRho` up to 50 cm
- S5: flat `|dxy|` up to 30 cm

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, branch, xlabel, xlim in [
    (axes[0], 'muonDxy', r'Gen $d_{xy}$ [cm]',                (-60, 60)),
    (axes[1], 'muonRho', r'Gen $\rho$ (transv. displ.) [cm]',  (0, 60)),
]:
    bins = np.linspace(xlim[0], xlim[1], 60)
    for tag in LOADED:
        lbl, col, ls, _ = DS_META[tag]
        ax.hist(data[tag][branch], bins=bins, histtype='step', label=lbl,
                color=col, linestyle=ls, linewidth=1.6, density=True)
    for xv, label in [(30, 'S5 max 30cm'), (50, 'S2/B2 max 50cm')]:
        for sign in ([1] if branch == 'muonRho' else [1, -1]):
            ax.axvline(sign * xv, color='tab:gray', linestyle=':', linewidth=0.9, alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_title(xlabel)
    ax.set_xlim(xlim)

axes[0].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('testset_04_displacement.pdf', bbox_inches='tight')
plt.show()

## 5. OMTF reconstruction — pT response

### 5a. 2D: OMTF pT vs Gen pT

In [ ]:
show_tags = [t for t in ['S1', 'B1', 'S2', 'B2'] if t in data]
fig, axes = plt.subplots(1, len(show_tags), figsize=(5.5 * len(show_tags), 5))
if len(show_tags) == 1:
    axes = [axes]

for ax, tag in zip(axes, show_tags):
    d = data[tag]
    mask = d['omtfPt'] > 0
    h = ax.hexbin(d['muonPt'][mask], d['omtfPt'][mask],
                  gridsize=35, cmap='plasma', mincnt=1,
                  extent=[0, 210, 0, 210])
    plt.colorbar(h, ax=ax, label='Counts')
    ax.plot([0, 210], [0, 210], 'w--', linewidth=1, label='1:1')
    ax.set_xlabel(r'Gen $p_T$ [GeV]')
    ax.set_ylabel(r'OMTF $p_T$ [GeV]')
    ax.set_title(f'{DS_META[tag][0]}\n(n={mask.sum()} matched)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('testset_05a_pt2d.pdf', bbox_inches='tight')
plt.show()

### 5b. pT resolution: (OMTF pT − Gen pT) / Gen pT

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    d = data[tag]
    mask = (d['omtfPt'] > 0) & (d['muonPt'] > 0)
    if mask.sum() < 5:
        continue
    res = (d['omtfPt'][mask] - d['muonPt'][mask]) / d['muonPt'][mask]
    ax.hist(res, bins=60, range=(-2, 5), histtype='step', label=lbl,
            color=col, linestyle=ls, linewidth=1.6, density=True)

ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel(r'$(p_T^{\rm OMTF} - p_T^{\rm gen}) / p_T^{\rm gen}$')
ax.set_ylabel('Density')
ax.set_title('OMTF pT resolution (matched candidates only)')
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig('testset_05b_resolution.pdf', bbox_inches='tight')
plt.show()

### 5c. Trigger efficiency turn-on curves

Efficiency = fraction of candidates with `omtfPt > threshold` as a function of Gen pT.  
**Expected**: turn-on plateau near threshold value.

In [ ]:
THRESHOLD = 22.0  # GeV
pt_edges   = np.array([0, 5, 10, 15, 18, 22, 26, 30, 40, 55, 75, 110, 160, 210])
pt_centers = 0.5 * (pt_edges[:-1] + pt_edges[1:])

fig, ax = plt.subplots(figsize=(9, 4))

for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    d = data[tag]
    effs, errs = [], []
    for lo, hi in zip(pt_edges[:-1], pt_edges[1:]):
        sel   = (d['muonPt'] >= lo) & (d['muonPt'] < hi)
        ntot  = sel.sum()
        npass = ((d['omtfPt'] >= THRESHOLD) & sel).sum()
        eff   = npass / ntot if ntot > 0 else np.nan
        err   = (np.sqrt(eff * (1-eff) / ntot)
                 if ntot > 0 and not np.isnan(eff) else 0)
        effs.append(eff);  errs.append(err)
    effs, errs = np.array(effs), np.array(errs)
    ok = ~np.isnan(effs)
    ax.errorbar(pt_centers[ok], effs[ok], yerr=errs[ok],
                label=lbl, color=col, linestyle=ls, linewidth=1.6,
                marker='o', markersize=4, capsize=3)

ax.axvline(THRESHOLD, color='gray', linestyle=':', label=f'{THRESHOLD} GeV threshold')
ax.axhline(1.0, color='lightgray', linewidth=0.8)
ax.set_xlabel(r'Gen $p_T$ [GeV]')
ax.set_ylabel(fr'$\varepsilon$ (OMTF $p_T$ > {THRESHOLD} GeV)')
ax.set_title(f'OMTF trigger turn-on (threshold = {THRESHOLD} GeV)')
ax.set_ylim(-0.05, 1.15)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('testset_05c_turnon.pdf', bbox_inches='tight')
plt.show()

## 6. OMTF quality, score, and fired layers

**omtfQuality**: integer 0–15, higher = better (more detector layers matched).  
**omtfFiredLayers**: bitmask over ~18 layers. Count of set bits = number of layers with hits.  
**omtfScore**: internal pattern score; higher = better match.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    axes[0].hist(data[tag]['omtfQuality'], bins=np.arange(-0.5, 17),
                 histtype='step', label=lbl, color=col, linestyle=ls,
                 linewidth=1.6, density=True)
axes[0].set_xlabel('OMTF quality')
axes[0].set_ylabel('Density')
axes[0].set_title('OMTF quality (0\u201315)')
axes[0].legend(ncol=2, fontsize=8)

all_scores = np.concatenate([data[t]['omtfScore'] for t in LOADED])
slo, shi = np.percentile(all_scores, 1), np.percentile(all_scores, 99)
for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    axes[1].hist(data[tag]['omtfScore'], bins=50, range=(slo, shi),
                 histtype='step', label=lbl, color=col, linestyle=ls,
                 linewidth=1.6, density=True)
axes[1].set_xlabel('OMTF score')
axes[1].set_ylabel('Density')
axes[1].set_title('OMTF pattern score')

for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    pc = np.array([bin(int(x)).count('1') for x in data[tag]['omtfFiredLayers']])
    axes[2].hist(pc, bins=np.arange(-0.5, 20),
                 histtype='step', label=lbl, color=col, linestyle=ls,
                 linewidth=1.6, density=True)
axes[2].set_xlabel('Fired layers (popcount of bitmask)')
axes[2].set_ylabel('Density')
axes[2].set_title('Number of fired OMTF layers')

plt.tight_layout()
plt.savefig('testset_06_quality.pdf', bbox_inches='tight')
plt.show()

## 7. Hit multiplicity and occupancy

**Expected**:
- PU200 datasets (B1, B2, B3) should have **higher hit counts** than their no-PU counterparts.
- Displaced muons (S2, S5, B2) may have **fewer detector layers** fired (hits farther from IP miss inner stations).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for tag in LOADED:
    lbl, col, ls, _ = DS_META[tag]
    ax.hist(data[tag]['nhits'], bins=np.arange(-0.5, 16),
            histtype='step',
            label=f'{lbl} (mean={data[tag]["nhits"].mean():.1f})',
            color=col, linestyle=ls, linewidth=1.8, density=True)

ax.set_xlabel('Hits per OMTF candidate')
ax.set_ylabel('Density')
ax.set_title('Hit multiplicity per OMTF candidate')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('testset_07_nhits.pdf', bbox_inches='tight')
plt.show()

## 8. Signal vs background comparison — S1 vs B1 (prompt) and S2 vs B2 (displaced)

Direct overlay to confirm PU200 effect on OMTF reconstruction.

In [ ]:
pairs = [(n, b, d) for n, b, d in [('S1','B1','prompt'), ('S2','B2','displaced'), ('S3','B3','di-\u03bc')]
         if n in data and b in data]

nrows = len(pairs)
fig, axes = plt.subplots(nrows, 3, figsize=(15, 4.5 * nrows))
if nrows == 1:
    axes = [axes]

for row, (noPU, PU, desc) in enumerate(pairs):
    all_scores = np.concatenate([data[noPU]['omtfScore'], data[PU]['omtfScore']])
    slo, shi = np.percentile(all_scores, 1), np.percentile(all_scores, 99)

    specs = [
        (axes[row][0], 'omtfPt',    r'OMTF $p_T$ [GeV]',      np.linspace(0, 200, 40)),
        (axes[row][1], 'nhits',     'N hits per candidate',     np.arange(-0.5, 16)),
        (axes[row][2], 'omtfScore', 'OMTF score',               np.linspace(slo, shi, 40)),
    ]
    for ax, branch, xlabel, bins in specs:
        for tag, pu_label in [(noPU, 'no PU'), (PU, 'PU200')]:
            lbl, col, ls, _ = DS_META[tag]
            ax.hist(data[tag][branch].astype(float), bins=bins, histtype='step',
                    label=f'{tag} ({pu_label})',
                    color=col, linestyle=ls, linewidth=1.8, density=True)
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Density')
        ax.set_title(f'{desc}: {xlabel}')
        ax.legend(fontsize=8)

plt.suptitle('No-PU vs PU200 comparison', fontsize=13)
plt.tight_layout()
plt.savefig('testset_08_PU_comparison.pdf', bbox_inches='tight')
plt.show()

## 9. Displacement effect on OMTF pT response

Compare `muonDxy` vs `(omtfPt − muonPt)/muonPt` to see if displaced tracks have worse pT resolution.

In [ ]:
disp_tags = [t for t in ['S2', 'S5', 'B2'] if t in data]
fig, axes = plt.subplots(1, len(disp_tags), figsize=(5.5 * len(disp_tags), 5))
if len(disp_tags) == 1:
    axes = [axes]

for ax, tag in zip(axes, disp_tags):
    d = data[tag]
    mask = (d['omtfPt'] > 0) & (d['muonPt'] > 0)
    dxy  = np.abs(d['muonDxy'][mask])
    res  = (d['omtfPt'][mask] - d['muonPt'][mask]) / d['muonPt'][mask]
    h = ax.hexbin(dxy, res, gridsize=25, cmap='viridis', mincnt=1,
                  extent=[0, 55, -2.5, 5])
    plt.colorbar(h, ax=ax, label='Counts')
    ax.axhline(0, color='white', linestyle='--', linewidth=1)
    ax.set_xlabel(r'$|d_{xy}|$ [cm]')
    ax.set_ylabel(r'$(p_T^{\rm OMTF} - p_T^{\rm gen}) / p_T^{\rm gen}$')
    ax.set_title(f'{DS_META[tag][0]}\npT resolution vs |dxy| (n={mask.sum()} matched)')

plt.tight_layout()
plt.savefig('testset_09_disp_vs_res.pdf', bbox_inches='tight')
plt.show()

## 10. OMTF matching: ΔEta, ΔPhi and killed fraction

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, branch, xlabel in [
    (axes[0], 'deltaEta', r'$\Delta\eta$ (gen \u2212 OMTF)'),
    (axes[1], 'deltaPhi', r'$\Delta\phi$ (gen \u2212 OMTF)'),
]:
    for tag in LOADED:
        lbl, col, ls, _ = DS_META[tag]
        ax.hist(data[tag][branch], bins=60, range=(-0.5, 0.5),
                histtype='step', label=lbl, color=col, linestyle=ls,
                linewidth=1.6, density=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_title(xlabel)

axes[0].legend(ncol=2, fontsize=8)

ax = axes[2]
kill_labels = [DS_META[t][0] for t in LOADED]
kill_fracs  = [data[t]['killed'].astype(float).mean() * 100 for t in LOADED]
kill_colors = [DS_META[t][1] for t in LOADED]
bars = ax.bar(kill_labels, kill_fracs, color=kill_colors, edgecolor='black', linewidth=0.7)
ax.bar_label(bars, fmt='%.1f%%', padding=2, fontsize=8)
ax.set_ylabel('Fraction killed [%]')
ax.set_title('OMTF track-killer fraction')
ax.set_ylim(0, max(kill_fracs) * 1.3 + 0.5)
plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('testset_10_matching.pdf', bbox_inches='tight')
plt.show()

## 11. Summary table

In [ ]:
hdr = (f"{'DS':<6} {'N':>6} {'GenPt':>8} {'|dxy|':>8} "
       f"{'OMTF>0':>8} {'OMTFPt':>8} {'nHits':>7} {'Killed':>8}")
print(hdr)
print('-' * len(hdr))

for tag in LOADED:
    d = data[tag]
    n      = len(d['muonPt'])
    pt_m   = d['muonPt'].mean()
    dxy_m  = np.abs(d['muonDxy']).mean()
    reco_f = 100 * np.mean(d['omtfPt'] > 0)
    opt_m  = d['omtfPt'][d['omtfPt'] > 0].mean() if np.any(d['omtfPt'] > 0) else 0.0
    nh_m   = d['nhits'].mean()
    kill_f = 100 * d['killed'].astype(float).mean()
    print(f"{tag:<6} {n:>6} {pt_m:>8.1f} {dxy_m:>8.3f} "
          f"{reco_f:>7.1f}% {opt_m:>8.1f} {nh_m:>7.2f} {kill_f:>7.1f}%")